# Notebook 6: TAG - Taxonomy Classification

## Learning Objectives:
- Load taxonomy from Excel
- Classify queries by genre
- Use domain knowledge for better recommendations

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results --quiet

In [1]:
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from typing import Literal

from ai_course_utils import load_llm_from_env, load_use_case_config, load_taxonomy_from_excel

In [2]:
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
config = load_use_case_config(use_case_file)
system_prompt = config.get('agent_prompt')

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url


## Load Taxonomy

**USER INPUT**: Provide path to taxonomy Excel

In [3]:
# ============================================================================
# USER INPUT: Specify taxonomy path
# ============================================================================
taxonomy_file = "../Data/Movie_Recommendation_Taxonomy_File.xlsx"

# Load taxonomy
taxonomy = load_taxonomy_from_excel(taxonomy_file)

print(f'\nAvailable Genres:')
for genre in taxonomy.keys():
    print(f'  - {genre}')

✓ Taxonomy loaded: ../Data/Movie_Recommendation_Taxonomy_File.xlsx
  Genres: 10 (Neo-Mythical, Futuristic Noir, Eco-Thriller...)

Available Genres:
  - Neo-Mythical
  - Futuristic Noir
  - Eco-Thriller
  - Historical Fantasy
  - Cyber-Western
  - Ballroom Dancing
  - Bollywood Historical
  - Harry Potter movies
  - Star Wars movies
  - Chick Flicks


## Test Taxonomy

In [4]:
# Show example genre
example_genre = list(taxonomy.keys())[0]
print(f'Genre: {example_genre}')
print(f'Description: {taxonomy[example_genre]["description"]}')
print(f'Guidelines: {taxonomy[example_genre]["guidelines"][:100]}...')

Genre: Neo-Mythical
Description: Films blending ancient mythology with modern settings or themes.
Guidelines: Select a movie that includes both Mythology and Modern setting and create a narrative to show how it...


## Define Tools with Genre Classification

In [5]:
@tool
def search_web(query: str) -> str:
    """Search web for movie info."""
    search = GoogleSerperAPIWrapper()
    return search.run(query)

@tool
def classify_genre(query: str) -> dict:
    """Classify query into movie genre.
    
    Returns genre name, description, and recommendation guidelines.
    Use this to understand user preferences and provide targeted recommendations.
    """
    query_lower = query.lower()
    
    for genre_name, info in taxonomy.items():
        included = info.get('included', '').lower()
        keywords = [k.strip() for k in included.split(',') if k.strip()]
        
        if any(kw in query_lower for kw in keywords):
            return {
                'genre': genre_name,
                'description': info.get('description', ''),
                'guidelines': info.get('guidelines', ''),
                'matched': True
            }
    
    return {
        'genre': 'General',
        'description': 'No specific genre match',
        'guidelines': 'Provide general recommendations',
        'matched': False
    }

tools = [search_web, classify_genre]
print(f'✓ Tools: {[t.name for t in tools]}')

✓ Tools: ['search_web', 'classify_genre']


In [6]:
llm = load_llm_from_env()
llm_with_tools = llm.bind_tools(tools)

🤖 Loading LLM: openai / gpt-4.1-mini


In [7]:
def assistant(state):
    return {'messages': [llm_with_tools.invoke([SystemMessage(content=system_prompt)] + state['messages'])]}

def should_continue(state):
    return 'tools' if state['messages'][-1].tool_calls else '__end__'

graph_builder = StateGraph(MessagesState)
graph_builder.add_node('assistant', assistant)
graph_builder.add_node('tools', ToolNode(tools))
graph_builder.add_edge(START, 'assistant')
graph_builder.add_conditional_edges('assistant', should_continue)
graph_builder.add_edge('tools', 'assistant')
graph = graph_builder.compile(checkpointer=MemorySaver())

print('✓ Graph with TAG built')

✓ Graph with TAG built


In [8]:
def chat(msg, tid='default'):
    for event in graph.stream({'messages': [HumanMessage(content=msg)]}, {'configurable': {'thread_id': tid}}, stream_mode='values'):
        pass
    return event['messages'][-1].content

print('🎬 TAG-enabled chatbot ready!')

🎬 TAG-enabled chatbot ready!


## Test Genre Classification

In [9]:
# Test with genre-specific query
query = 'I love movies with mythology and sci-fi'
print(f'User: {query}')
response = chat(query)
print(f'Bot: {response}')

User: I love movies with mythology and sci-fi
Bot: Great combination! Are you more interested in movies that lean heavily into mythology, or those that are more focused on sci-fi elements? Also, do you have a preference for a particular type of mythology (like Greek, Norse, Egyptian, etc.) or a specific sci-fi theme (space exploration, time travel, futuristic technology)? And finally, are you looking for something recent or from a particular decade?


In [ ]:
# Another genre test
query2 = 'Suggest a cyberpunk western'
print(f'\nUser: {query2}')
response2 = chat(query2)
print(f'Bot: {response2}')

## Summary

### What We Built:
✅ Taxonomy-based genre classification  
✅ Domain knowledge integration  
✅ Smarter query understanding  
✅ Genre-specific recommendations  

### Your Taxonomy Has 10 Genres:
- Neo-Mythical
- Futuristic Noir
- Eco-Thriller
- Historical Fantasy
- Cyber-Western
- Ballroom Dancing
- Bollywood Historical
- Harry Potter movies
- Star Wars movies
- Chick Flicks

### Next: Notebook 7 - Complete System (RAG + TAG)